# 🎯 Adversarial Corruption Hyperparameter Tuning

This notebook performs comprehensive hyperparameter optimization for adversarial corruption in energy-based models using Weights & Biases.

## 🔧 Hyperparameters to Optimize:
- **Warmup Steps**: Number of steps before adversarial corruption begins
- **Adversarial Steps**: Number of gradient steps for adversarial noise generation
- **Distance Penalty**: Weight for constraining adversarial perturbations

## 📊 Optimization Strategy:
- Bayesian optimization with early termination
- Multi-objective evaluation (accuracy, loss, stability)
- Comprehensive visualization and analysis

## 🚀 Environment Setup

In [ ]:
# Check if we're running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("🔵 Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("🟡 Running in local environment")

# Mount Google Drive if needed (optional)
if IN_COLAB:
    from google.colab import drive
    # drive.mount('/content/drive')  # Uncomment if you want to save results to Drive

In [ ]:
# Clone the energy-based model repository
!rm -rf energy-based-model
!git clone https://github.com/mdkrasnow/energy-based-model.git
%cd energy-based-model

# Verify we're in the right directory
!ls -la

In [ ]:
# Install all required dependencies
!pip install -q torch torchvision einops accelerate tqdm tabulate matplotlib numpy pandas ema-pytorch ipdb
!pip install -q wandb plotly seaborn scikit-learn

print("✅ All dependencies installed successfully!")

In [ ]:
# Import all necessary libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb
import os
import json
import yaml
import subprocess
import time
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('default')
sns.set_palette("husl")

print("📚 All libraries imported successfully!")

## 🔑 Weights & Biases Setup

In [ ]:
# Initialize Weights & Biases
# You'll need to enter your API key when prompted
print("🔐 Please log in to Weights & Biases...")
print("If you don't have an account, create one at https://wandb.ai")

wandb.login()

print("✅ Successfully logged in to Weights & Biases!")

In [ ]:
# Create the Weights & Biases sweep configuration
SWEEP_CONFIG = {
    'method': 'bayes',  # Bayesian optimization
    'metric': {
        'goal': 'maximize',
        'name': 'final_accuracy'
    },
    'parameters': {
        'anm_warmup_steps': {
            'values': [100, 250, 500, 750, 1000, 1250, 1500]
        },
        'anm_adversarial_steps': {
            'values': [1, 2, 3, 4, 5, 6, 8]
        },
        'anm_distance_penalty': {
            'distribution': 'log_uniform_values',
            'values': [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
        }
    },
    'early_terminate': {
        'type': 'hyperband',
        'min_iter': 3
    }
}

# Save configuration to file
with open('wandb_sweep_config.yaml', 'w') as f:
    yaml.dump(SWEEP_CONFIG, f, default_flow_style=False)

print("📄 Sweep configuration created:")
print(yaml.dump(SWEEP_CONFIG, default_flow_style=False))

## 🧠 Training Configuration & Helper Functions

In [ ]:
# Base training configuration
BASE_CONFIG = {
    'dataset': 'inverse',
    'model': 'mlp', 
    'batch_size': 32,
    'diffusion_steps': 10,
    'supervise_energy_landscape': True,
    'train_num_steps': 2000,  # Increased for better optimization
    'save_csv_logs': True,
    'csv_log_interval': 100,
    'use_adversarial_corruption': True
}

# Create results directory
RESULTS_DIR = Path('./hyperparameter_results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f"📁 Results will be saved to: {RESULTS_DIR.absolute()}")
print(f"🎯 Base configuration: {BASE_CONFIG}")

In [ ]:
def parse_training_output(output: str) -> Dict[str, float]:
    """
    Parse training output to extract key metrics.
    """
    metrics = {}
    
    # Look for final loss values in the output
    patterns = {
        'final_total_loss': r'total_loss[:\s]+(\d+\.\d+)',
        'final_energy_loss': r'loss_energy[:\s]+(\d+\.\d+)', 
        'final_denoise_loss': r'loss_denoise[:\s]+(\d+\.\d+)'
    }
    
    for key, pattern in patterns.items():
        matches = re.findall(pattern, output)
        if matches:
            metrics[key] = float(matches[-1])  # Take the last occurrence
    
    return metrics


def load_latest_csv(csv_dir: str, pattern: str) -> Optional[pd.DataFrame]:
    """
    Load the most recent CSV file matching the pattern.
    """
    csv_path = Path(csv_dir)
    if not csv_path.exists():
        return None
        
    files = list(csv_path.glob(pattern))
    if not files:
        return None
        
    latest_file = max(files, key=lambda x: x.stat().st_mtime)
    return pd.read_csv(latest_file)


def extract_accuracy_from_validation(val_df: Optional[pd.DataFrame], 
                                    dataset_type: str = 'inverse') -> float:
    """
    Extract final accuracy from validation data.
    """
    if val_df is None or len(val_df) == 0:
        return 0.0
    
    if dataset_type == 'inverse':
        # For MSE-based tasks, convert MSE to accuracy
        mse_data = val_df[val_df['metric_name'] == 'mse_error']
        if len(mse_data) == 0:
            # Try alternative name 'mse' if 'mse_error' not found
            mse_data = val_df[val_df['metric_name'] == 'mse']
        
        if len(mse_data) > 0:
            # Convert metric_value to float to handle string values from CSV
            final_mse = float(mse_data['metric_value'].iloc[-1])
            # Convert MSE to accuracy using exponential decay
            accuracy = np.exp(-final_mse) * 100
            return min(accuracy, 100.0)  # Cap at 100%
    
    elif dataset_type in ['bce', 'connectivity', 'parents']:
        # For classification tasks, use accuracy directly
        acc_data = val_df[val_df['metric_name'].str.contains('accuracy', na=False)]
        if len(acc_data) > 0:
            return float(acc_data['metric_value'].iloc[-1]) * 100
    
    return 0.0


def calculate_training_stability(train_df: Optional[pd.DataFrame]) -> float:
    """
    Calculate training stability based on loss variance.
    """
    if train_df is None or len(train_df) < 2:
        return 0.0
    
    # Use the last 50% of training for stability calculation
    second_half = train_df[len(train_df)//2:]
    
    if 'total_loss' in second_half.columns:
        loss_std = second_half['total_loss'].std()
        loss_mean = second_half['total_loss'].mean()
        
        # Coefficient of variation (lower is better)
        if loss_mean > 0:
            cv = loss_std / loss_mean
            # Convert to stability score (higher is better)
            stability = max(0, 1 - cv)
            return stability * 100
    
    return 0.0


print("🔧 Helper functions defined successfully!")

## 🎯 Main Training Function

In [ ]:
def train_with_hyperparameters(config=None):
    """
    Main training function called by wandb sweep.
    """
    # Initialize wandb run
    with wandb.init(config=config) as run:
        # Get hyperparameters from wandb config
        config = wandb.config
        
        # Extract hyperparameters
        warmup_steps = config.anm_warmup_steps
        adversarial_steps = config.anm_adversarial_steps  
        distance_penalty = config.anm_distance_penalty
        
        # Create unique directory for this run
        run_dir = RESULTS_DIR / f"run_{run.id}"
        run_dir.mkdir(exist_ok=True)
        
        print(f"🚀 Starting training run {run.id}")
        print(f"📊 Hyperparameters: warmup={warmup_steps}, adv_steps={adversarial_steps}, penalty={distance_penalty}")
        
        # Build training command
        cmd = [
            'python', 'train.py',
            '--dataset', BASE_CONFIG['dataset'],
            '--model', BASE_CONFIG['model'],
            '--batch-size', str(BASE_CONFIG['batch_size']),
            '--diffusion-steps', str(BASE_CONFIG['diffusion_steps']),
            '--supervise-energy-landscape', str(BASE_CONFIG['supervise_energy_landscape']),
            '--train-num-steps', str(BASE_CONFIG['train_num_steps']),
            '--save-csv-logs',
            '--csv-log-interval', str(BASE_CONFIG['csv_log_interval']),
            '--csv-log-dir', str(run_dir),
            '--use-adversarial-corruption', 'True',
            '--anm-warmup-steps', str(warmup_steps),
            '--anm-adversarial-steps', str(adversarial_steps),
            '--anm-distance-penalty', str(distance_penalty)
        ]
        
        try:
            # Run training
            start_time = time.time()
            result = subprocess.run(
                cmd, 
                capture_output=True, 
                text=True, 
                cwd=os.getcwd(),
                timeout=1800  # 30 minute timeout
            )
            training_time = time.time() - start_time
            
            if result.returncode != 0:
                print(f"❌ Training failed with return code {result.returncode}")
                print(f"Error output: {result.stderr}")
                wandb.log({
                    'training_failed': True,
                    'final_accuracy': 0.0,
                    'final_total_loss': float('inf'),
                    'training_time': training_time
                })
                return
            
            print(f"✅ Training completed in {training_time:.1f}s")
            
            # Load and process results
            train_df = load_latest_csv(run_dir, 'training_metrics_*.csv')
            val_df = load_latest_csv(run_dir, 'validation_metrics_*.csv')
            energy_df = load_latest_csv(run_dir, 'energy_metrics_*.csv')
            
            # Extract metrics
            metrics = {
                'training_time': training_time,
                'warmup_steps': warmup_steps,
                'adversarial_steps': adversarial_steps,
                'distance_penalty': distance_penalty
            }
            
            # Process training metrics
            if train_df is not None and len(train_df) > 0:
                metrics.update({
                    'final_total_loss': train_df['total_loss'].iloc[-1],
                    'final_energy_loss': train_df['loss_energy'].iloc[-1],
                    'final_denoise_loss': train_df['loss_denoise'].iloc[-1],
                    'min_total_loss': train_df['total_loss'].min(),
                    'loss_reduction': (train_df['total_loss'].iloc[0] - train_df['total_loss'].iloc[-1]) / train_df['total_loss'].iloc[0] * 100,
                    'training_stability': calculate_training_stability(train_df)
                })
                
                # Log training curves
                for idx, row in train_df.iterrows():
                    wandb.log({
                        'step': row['step'],
                        'total_loss': row['total_loss'],
                        'energy_loss': row['loss_energy'],
                        'denoise_loss': row['loss_denoise']
                    }, step=row['step'])
            
            # Process validation metrics
            accuracy = extract_accuracy_from_validation(val_df, BASE_CONFIG['dataset'])
            metrics['final_accuracy'] = accuracy
            
            if val_df is not None:
                # Log validation curves
                for metric_name in val_df['metric_name'].unique():
                    metric_data = val_df[val_df['metric_name'] == metric_name]
                    for idx, row in metric_data.iterrows():
                        wandb.log({
                            f'val_{metric_name}': row['metric_value']
                        }, step=row['step'])
            
            # Process energy metrics
            if energy_df is not None and len(energy_df) > 0:
                metrics.update({
                    'max_curriculum_weight': energy_df['curriculum_weight'].max(),
                    'final_curriculum_weight': energy_df['curriculum_weight'].iloc[-1]
                })
                
                # Count corruption types
                corruption_counts = energy_df['corruption_type'].value_counts().to_dict()
                for corruption_type, count in corruption_counts.items():
                    metrics[f'corruption_{corruption_type}_count'] = count
            
            # Calculate composite score
            composite_score = (
                metrics.get('final_accuracy', 0) * 0.5 +  # 50% weight on accuracy
                metrics.get('training_stability', 0) * 0.3 +  # 30% weight on stability
                max(0, 100 - metrics.get('final_total_loss', 100)) * 0.2  # 20% weight on loss
            )
            metrics['composite_score'] = composite_score
            
            # Log all metrics to wandb
            wandb.log(metrics)
            
            print(f"📊 Final metrics: Accuracy={accuracy:.2f}%, Loss={metrics.get('final_total_loss', 'N/A'):.4f}, Score={composite_score:.2f}")
            
        except subprocess.TimeoutExpired:
            print("⏰ Training timed out")
            wandb.log({
                'training_timeout': True,
                'final_accuracy': 0.0,
                'final_total_loss': float('inf')
            })
        except Exception as e:
            print(f"💥 Training failed with exception: {str(e)}")
            wandb.log({
                'training_error': True,
                'error_message': str(e),
                'final_accuracy': 0.0,
                'final_total_loss': float('inf')
            })

print("🎯 Training function defined successfully!")

## 🌪️ Execute Hyperparameter Sweep

In [ ]:
# Initialize the sweep
PROJECT_NAME = "adversarial-corruption-tuning"

print(f"🌪️ Creating hyperparameter sweep in project: {PROJECT_NAME}")
print(f"🎯 Optimization target: {SWEEP_CONFIG['metric']['name']} ({SWEEP_CONFIG['metric']['goal']})")

sweep_id = wandb.sweep(
    sweep=SWEEP_CONFIG, 
    project=PROJECT_NAME
)

print(f"✅ Sweep created with ID: {sweep_id}")
print(f"🔗 View sweep at: https://wandb.ai/your-username/{PROJECT_NAME}/sweeps/{sweep_id}")

In [ ]:
# Run the sweep
print("🚀 Starting hyperparameter sweep...")
print("This will run multiple training jobs with different hyperparameter combinations.")
print("⏱️ Estimated time: 30-60 minutes depending on the number of runs")

# Set the number of runs for the sweep
NUM_SWEEP_RUNS = 20  # Adjust this based on your time/compute budget

print(f"🔄 Will execute {NUM_SWEEP_RUNS} hyperparameter combinations")

# Run the sweep agent
wandb.agent(
    sweep_id, 
    function=train_with_hyperparameters, 
    count=NUM_SWEEP_RUNS,
    project=PROJECT_NAME
)

print("🏁 Hyperparameter sweep completed!")

## 📊 Results Analysis & Visualization

In [ ]:
# Fetch results from wandb
print("📥 Fetching results from Weights & Biases...")

api = wandb.Api()
runs = api.runs(f"{PROJECT_NAME}", filters={"sweep": sweep_id})

# Collect all run data
results_data = []

for run in runs:
    if run.state == 'finished':  # Only include completed runs
        run_data = {
            'run_id': run.id,
            'run_name': run.name,
            'state': run.state,
            'warmup_steps': run.config.get('anm_warmup_steps'),
            'adversarial_steps': run.config.get('anm_adversarial_steps'),
            'distance_penalty': run.config.get('anm_distance_penalty')
        }
        
        # Add summary metrics
        for key, value in run.summary.items():
            if not key.startswith('_'):
                run_data[key] = value
        
        results_data.append(run_data)

# Create DataFrame
results_df = pd.DataFrame(results_data)

print(f"📊 Collected data from {len(results_df)} completed runs")
print(f"🔍 Available metrics: {list(results_df.columns)}")

# Display basic statistics
if len(results_df) > 0:
    print("\n📈 Quick Statistics:")
    print(f"  Best Accuracy: {results_df['final_accuracy'].max():.2f}%")
    print(f"  Mean Accuracy: {results_df['final_accuracy'].mean():.2f}%")
    print(f"  Best Loss: {results_df['final_total_loss'].min():.4f}")
    print(f"  Mean Loss: {results_df['final_total_loss'].mean():.4f}")
else:
    print("⚠️ No completed runs found. Please wait for the sweep to finish.")

In [ ]:
# Find and display best hyperparameters
if len(results_df) > 0:
    # Find best run by accuracy
    best_run_idx = results_df['final_accuracy'].idxmax()
    best_run = results_df.loc[best_run_idx]
    
    print("🏆 BEST HYPERPARAMETERS (by accuracy):")
    print("="*50)
    print(f"🔍 Run ID: {best_run['run_id']}")
    print(f"📊 Final Accuracy: {best_run['final_accuracy']:.2f}%")
    print(f"📉 Final Loss: {best_run['final_total_loss']:.4f}")
    print(f"⚙️ Warmup Steps: {best_run['warmup_steps']}")
    print(f"⚙️ Adversarial Steps: {best_run['adversarial_steps']}")
    print(f"⚙️ Distance Penalty: {best_run['distance_penalty']}")
    if 'training_stability' in best_run:
        print(f"📈 Training Stability: {best_run['training_stability']:.2f}%")
    if 'composite_score' in best_run:
        print(f"🎯 Composite Score: {best_run['composite_score']:.2f}")
    
    # Find best run by composite score if available
    if 'composite_score' in results_df.columns:
        best_composite_idx = results_df['composite_score'].idxmax()
        best_composite = results_df.loc[best_composite_idx]
        
        print("\n🎯 BEST HYPERPARAMETERS (by composite score):")
        print("="*50)
        print(f"🔍 Run ID: {best_composite['run_id']}")
        print(f"🎯 Composite Score: {best_composite['composite_score']:.2f}")
        print(f"📊 Final Accuracy: {best_composite['final_accuracy']:.2f}%")
        print(f"📉 Final Loss: {best_composite['final_total_loss']:.4f}")
        print(f"⚙️ Warmup Steps: {best_composite['warmup_steps']}")
        print(f"⚙️ Adversarial Steps: {best_composite['adversarial_steps']}")
        print(f"⚙️ Distance Penalty: {best_composite['distance_penalty']}")
    
    # Top 5 runs
    print("\n🥇 TOP 5 RUNS (by accuracy):")
    print("="*70)
    top_5 = results_df.nlargest(5, 'final_accuracy')[[
        'run_id', 'final_accuracy', 'final_total_loss', 
        'warmup_steps', 'adversarial_steps', 'distance_penalty'
    ]]
    for idx, row in top_5.iterrows():
        print(f"#{top_5.index.get_loc(idx)+1}: Acc={row['final_accuracy']:.2f}% | "
              f"Loss={row['final_total_loss']:.4f} | "
              f"W={row['warmup_steps']} | A={row['adversarial_steps']} | P={row['distance_penalty']:.3f}")

In [ ]:
# Create comprehensive visualizations
if len(results_df) > 0:
    # Set up the plotting style
    plt.style.use('default')
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('🎯 Hyperparameter Analysis: Adversarial Corruption Tuning', fontsize=16, fontweight='bold')
    
    # 1. Accuracy vs Warmup Steps
    scatter = axes[0, 0].scatter(results_df['warmup_steps'], results_df['final_accuracy'], 
                                c=results_df['distance_penalty'], cmap='viridis', 
                                s=results_df['adversarial_steps']*20, alpha=0.7)
    axes[0, 0].set_xlabel('Warmup Steps')
    axes[0, 0].set_ylabel('Final Accuracy (%)')
    axes[0, 0].set_title('Accuracy vs Warmup Steps\n(size=adv_steps, color=penalty)')
    axes[0, 0].grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=axes[0, 0], label='Distance Penalty')
    
    # 2. Accuracy vs Adversarial Steps
    scatter2 = axes[0, 1].scatter(results_df['adversarial_steps'], results_df['final_accuracy'],
                                 c=results_df['warmup_steps'], cmap='plasma',
                                 s=results_df['distance_penalty']*200, alpha=0.7)
    axes[0, 1].set_xlabel('Adversarial Steps')
    axes[0, 1].set_ylabel('Final Accuracy (%)')
    axes[0, 1].set_title('Accuracy vs Adversarial Steps\n(size=penalty, color=warmup)')
    axes[0, 1].grid(True, alpha=0.3)
    plt.colorbar(scatter2, ax=axes[0, 1], label='Warmup Steps')
    
    # 3. Accuracy vs Distance Penalty
    scatter3 = axes[0, 2].scatter(results_df['distance_penalty'], results_df['final_accuracy'],
                                 c=results_df['adversarial_steps'], cmap='coolwarm',
                                 s=results_df['warmup_steps']/10, alpha=0.7)
    axes[0, 2].set_xlabel('Distance Penalty')
    axes[0, 2].set_ylabel('Final Accuracy (%)')
    axes[0, 2].set_title('Accuracy vs Distance Penalty\n(size=warmup, color=adv_steps)')
    axes[0, 2].set_xscale('log')
    axes[0, 2].grid(True, alpha=0.3)
    plt.colorbar(scatter3, ax=axes[0, 2], label='Adversarial Steps')
    
    # 4. Loss vs Accuracy (with best runs highlighted)
    scatter4 = axes[1, 0].scatter(results_df['final_total_loss'], results_df['final_accuracy'],
                                 c=results_df['distance_penalty'], cmap='viridis', alpha=0.7)
    # Highlight best runs
    top_3 = results_df.nlargest(3, 'final_accuracy')
    axes[1, 0].scatter(top_3['final_total_loss'], top_3['final_accuracy'],
                      color='red', s=100, marker='*', label='Top 3 Runs')
    axes[1, 0].set_xlabel('Final Total Loss')
    axes[1, 0].set_ylabel('Final Accuracy (%)')
    axes[1, 0].set_title('Accuracy vs Loss Trade-off')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    plt.colorbar(scatter4, ax=axes[1, 0], label='Distance Penalty')
    
    # 5. Hyperparameter Correlation Heatmap
    param_cols = ['warmup_steps', 'adversarial_steps', 'distance_penalty', 
                  'final_accuracy', 'final_total_loss']
    if 'training_stability' in results_df.columns:
        param_cols.append('training_stability')
    
    corr_matrix = results_df[param_cols].corr()
    im = axes[1, 1].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[1, 1].set_xticks(range(len(param_cols)))
    axes[1, 1].set_yticks(range(len(param_cols)))
    axes[1, 1].set_xticklabels([col.replace('_', '\n') for col in param_cols], rotation=45)
    axes[1, 1].set_yticklabels([col.replace('_', '\n') for col in param_cols])
    axes[1, 1].set_title('Parameter Correlation Matrix')
    
    # Add correlation values to heatmap
    for i in range(len(param_cols)):
        for j in range(len(param_cols)):
            text = axes[1, 1].text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                                  ha="center", va="center", color="black" if abs(corr_matrix.iloc[i, j]) < 0.5 else "white")
    plt.colorbar(im, ax=axes[1, 1], label='Correlation')
    
    # 6. Training Time vs Performance
    if 'training_time' in results_df.columns:
        scatter6 = axes[1, 2].scatter(results_df['training_time'], results_df['final_accuracy'],
                                     c=results_df['adversarial_steps'], cmap='viridis', alpha=0.7)
        axes[1, 2].set_xlabel('Training Time (seconds)')
        axes[1, 2].set_ylabel('Final Accuracy (%)')
        axes[1, 2].set_title('Training Time vs Accuracy')
        axes[1, 2].grid(True, alpha=0.3)
        plt.colorbar(scatter6, ax=axes[1, 2], label='Adversarial Steps')
    else:
        # Alternative: Show distribution of best hyperparameter
        best_param = results_df.loc[results_df['final_accuracy'].idxmax(), 'warmup_steps']
        axes[1, 2].hist(results_df['warmup_steps'], bins=10, alpha=0.7, color='skyblue')
        axes[1, 2].axvline(best_param, color='red', linestyle='--', label=f'Best: {best_param}')
        axes[1, 2].set_xlabel('Warmup Steps')
        axes[1, 2].set_ylabel('Frequency')
        axes[1, 2].set_title('Distribution of Warmup Steps')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Save the plot
    plt.savefig(RESULTS_DIR / 'hyperparameter_analysis.png', dpi=300, bbox_inches='tight')
    print(f"💾 Visualization saved to {RESULTS_DIR / 'hyperparameter_analysis.png'}")

In [ ]:
# Create interactive 3D visualization with Plotly
if len(results_df) > 0:
    print("🎨 Creating interactive 3D visualization...")
    
    # Create 3D scatter plot
    fig = go.Figure(data=go.Scatter3d(
        x=results_df['warmup_steps'],
        y=results_df['adversarial_steps'],
        z=results_df['distance_penalty'],
        mode='markers',
        marker=dict(
            size=8,
            color=results_df['final_accuracy'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Final Accuracy (%)"),
            opacity=0.8
        ),
        text=[
            f'Run: {row["run_id"]}\n'
            f'Accuracy: {row["final_accuracy"]:.2f}%\n'
            f'Loss: {row["final_total_loss"]:.4f}\n'
            f'Warmup: {row["warmup_steps"]}\n'
            f'Adv Steps: {row["adversarial_steps"]}\n'
            f'Penalty: {row["distance_penalty"]:.3f}'
            for _, row in results_df.iterrows()
        ],
        hoverinfo='text'
    ))
    
    # Highlight best run
    best_run = results_df.loc[results_df['final_accuracy'].idxmax()]
    fig.add_trace(go.Scatter3d(
        x=[best_run['warmup_steps']],
        y=[best_run['adversarial_steps']],
        z=[best_run['distance_penalty']],
        mode='markers',
        marker=dict(
            size=15,
            color='red',
            symbol='diamond'
        ),
        name='Best Run',
        text=f'BEST RUN\nAccuracy: {best_run["final_accuracy"]:.2f}%'
    ))
    
    fig.update_layout(
        title={
            'text': '🎯 3D Hyperparameter Space Exploration',
            'x': 0.5,
            'font': {'size': 20}
        },
        scene=dict(
            xaxis_title='Warmup Steps',
            yaxis_title='Adversarial Steps',
            zaxis_title='Distance Penalty (log scale)',
            zaxis_type='log'
        ),
        width=900,
        height=700
    )
    
    fig.show()
    
    # Save interactive plot
    fig.write_html(str(RESULTS_DIR / '3d_hyperparameter_space.html'))
    print(f"💾 Interactive 3D plot saved to {RESULTS_DIR / '3d_hyperparameter_space.html'}")

In [ ]:
# Create parallel coordinates plot
if len(results_df) > 0:
    print("📊 Creating parallel coordinates visualization...")
    
    # Prepare data for parallel coordinates
    plot_df = results_df.copy()
    
    # Normalize distance penalty for better visualization
    plot_df['log_distance_penalty'] = np.log10(plot_df['distance_penalty'])
    
    # Create parallel coordinates plot
    fig = go.Figure(data=go.Parcoords(
        line=dict(
            color=plot_df['final_accuracy'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Final Accuracy (%)")
        ),
        dimensions=[
            dict(
                range=[plot_df['warmup_steps'].min(), plot_df['warmup_steps'].max()],
                label='Warmup Steps',
                values=plot_df['warmup_steps']
            ),
            dict(
                range=[plot_df['adversarial_steps'].min(), plot_df['adversarial_steps'].max()],
                label='Adversarial Steps',
                values=plot_df['adversarial_steps']
            ),
            dict(
                range=[plot_df['log_distance_penalty'].min(), plot_df['log_distance_penalty'].max()],
                label='log₁₀(Distance Penalty)',
                values=plot_df['log_distance_penalty']
            ),
            dict(
                range=[plot_df['final_accuracy'].min(), plot_df['final_accuracy'].max()],
                label='Final Accuracy (%)',
                values=plot_df['final_accuracy']
            ),
            dict(
                range=[plot_df['final_total_loss'].min(), plot_df['final_total_loss'].max()],
                label='Final Loss',
                values=plot_df['final_total_loss']
            )
        ]
    ))
    
    fig.update_layout(
        title={
            'text': '📊 Parallel Coordinates: Hyperparameter → Performance Mapping',
            'x': 0.5,
            'font': {'size': 18}
        },
        width=1000,
        height=600
    )
    
    fig.show()
    
    # Save parallel coordinates plot
    fig.write_html(str(RESULTS_DIR / 'parallel_coordinates.html'))
    print(f"💾 Parallel coordinates plot saved to {RESULTS_DIR / 'parallel_coordinates.html'}")

## 🔍 Detailed Statistical Analysis

In [ ]:
# Perform detailed statistical analysis
if len(results_df) > 0:
    print("🔍 DETAILED STATISTICAL ANALYSIS")
    print("="*60)
    
    # Basic statistics
    print("\n📊 DESCRIPTIVE STATISTICS:")
    print("-"*40)
    
    stats_cols = ['warmup_steps', 'adversarial_steps', 'distance_penalty', 
                  'final_accuracy', 'final_total_loss']
    
    for col in stats_cols:
        if col in results_df.columns:
            data = results_df[col]
            print(f"{col.replace('_', ' ').title()}:")
            print(f"  Mean: {data.mean():.4f}")
            print(f"  Std:  {data.std():.4f}")
            print(f"  Min:  {data.min():.4f}")
            print(f"  Max:  {data.max():.4f}")
            print(f"  Q25:  {data.quantile(0.25):.4f}")
            print(f"  Q75:  {data.quantile(0.75):.4f}")
            print()
    
    # Identify parameter ranges for top performers
    print("\n🏆 TOP 25% PERFORMERS ANALYSIS:")
    print("-"*40)
    
    top_25_percent = results_df.nlargest(max(1, len(results_df)//4), 'final_accuracy')
    
    print(f"Number of top performers: {len(top_25_percent)}")
    print(f"Accuracy range: {top_25_percent['final_accuracy'].min():.2f}% - {top_25_percent['final_accuracy'].max():.2f}%")
    print()
    
    for param in ['warmup_steps', 'adversarial_steps', 'distance_penalty']:
        param_data = top_25_percent[param]
        print(f"{param.replace('_', ' ').title()} (top 25%):")
        print(f"  Range: {param_data.min()} - {param_data.max()}")
        print(f"  Mean: {param_data.mean():.3f}")
        print(f"  Most common: {param_data.mode().iloc[0] if len(param_data.mode()) > 0 else 'N/A'}")
        print()
    
    # Parameter sensitivity analysis
    print("\n📈 PARAMETER SENSITIVITY:")
    print("-"*40)
    
    for param in ['warmup_steps', 'adversarial_steps', 'distance_penalty']:
        correlation = results_df[param].corr(results_df['final_accuracy'])
        print(f"{param.replace('_', ' ').title()} vs Accuracy correlation: {correlation:.3f}")
        
        if abs(correlation) > 0.3:
            direction = "positively" if correlation > 0 else "negatively"
            strength = "strongly" if abs(correlation) > 0.6 else "moderately"
            print(f"  → {strength} {direction} correlated with accuracy")
        else:
            print(f"  → weakly correlated with accuracy")
        print()
    
    # Interaction effects
    print("\n🔄 PARAMETER INTERACTIONS:")
    print("-"*40)
    
    # Check if there are obvious interaction patterns
    warmup_high = results_df['warmup_steps'] > results_df['warmup_steps'].median()
    adv_high = results_df['adversarial_steps'] > results_df['adversarial_steps'].median()
    penalty_high = results_df['distance_penalty'] > results_df['distance_penalty'].median()
    
    interaction_groups = {
        'High Warmup + High Adv Steps': results_df[warmup_high & adv_high]['final_accuracy'].mean(),
        'High Warmup + Low Adv Steps': results_df[warmup_high & ~adv_high]['final_accuracy'].mean(),
        'Low Warmup + High Adv Steps': results_df[~warmup_high & adv_high]['final_accuracy'].mean(),
        'Low Warmup + Low Adv Steps': results_df[~warmup_high & ~adv_high]['final_accuracy'].mean(),
    }
    
    for group, avg_acc in interaction_groups.items():
        if not np.isnan(avg_acc):
            print(f"{group}: {avg_acc:.2f}% average accuracy")
    
    # Recommendations
    print("\n💡 RECOMMENDATIONS:")
    print("-"*40)
    
    best_params = results_df.loc[results_df['final_accuracy'].idxmax()]
    
    print(f"🎯 For maximum accuracy, use:")
    print(f"   Warmup Steps: {best_params['warmup_steps']}")
    print(f"   Adversarial Steps: {best_params['adversarial_steps']}")
    print(f"   Distance Penalty: {best_params['distance_penalty']:.3f}")
    
    # Find robust parameters (good performance across multiple runs)
    print(f"\n🛡️ For robust performance, consider ranges:")
    for param in ['warmup_steps', 'adversarial_steps', 'distance_penalty']:
        param_range = (top_25_percent[param].min(), top_25_percent[param].max())
        print(f"   {param.replace('_', ' ').title()}: {param_range[0]} - {param_range[1]}")
    
    print("\n✅ Analysis complete!")

In [ ]:
# Save comprehensive results
if len(results_df) > 0:
    print("💾 Saving comprehensive results...")
    
    # Save raw results
    results_df.to_csv(RESULTS_DIR / 'hyperparameter_sweep_results.csv', index=False)
    print(f"📊 Raw results saved to {RESULTS_DIR / 'hyperparameter_sweep_results.csv'}")
    
    # Save best hyperparameters
    best_params_summary = {
        'best_run_id': results_df.loc[results_df['final_accuracy'].idxmax(), 'run_id'],
        'best_accuracy': results_df['final_accuracy'].max(),
        'best_warmup_steps': int(results_df.loc[results_df['final_accuracy'].idxmax(), 'warmup_steps']),
        'best_adversarial_steps': int(results_df.loc[results_df['final_accuracy'].idxmax(), 'adversarial_steps']),
        'best_distance_penalty': float(results_df.loc[results_df['final_accuracy'].idxmax(), 'distance_penalty']),
        'sweep_id': sweep_id,
        'total_runs': len(results_df),
        'timestamp': datetime.now().isoformat()
    }
    
    with open(RESULTS_DIR / 'best_hyperparameters.json', 'w') as f:
        json.dump(best_params_summary, f, indent=2)
    print(f"🏆 Best hyperparameters saved to {RESULTS_DIR / 'best_hyperparameters.json'}")
    
    # Create summary report
    summary_report = f"""
🎯 ADVERSARIAL CORRUPTION HYPERPARAMETER TUNING REPORT
═══════════════════════════════════════════════════════

Sweep ID: {sweep_id}
Project: {PROJECT_NAME}
Total Runs: {len(results_df)}
Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

🏆 BEST CONFIGURATION:
├─ Accuracy: {results_df['final_accuracy'].max():.2f}%
├─ Total Loss: {results_df.loc[results_df['final_accuracy'].idxmax(), 'final_total_loss']:.4f}
├─ Warmup Steps: {int(results_df.loc[results_df['final_accuracy'].idxmax(), 'warmup_steps'])}
├─ Adversarial Steps: {int(results_df.loc[results_df['final_accuracy'].idxmax(), 'adversarial_steps'])}
└─ Distance Penalty: {float(results_df.loc[results_df['final_accuracy'].idxmax(), 'distance_penalty']):.3f}

📊 PERFORMANCE SUMMARY:
├─ Mean Accuracy: {results_df['final_accuracy'].mean():.2f}%
├─ Std Accuracy: {results_df['final_accuracy'].std():.2f}%
├─ Best Accuracy: {results_df['final_accuracy'].max():.2f}%
├─ Worst Accuracy: {results_df['final_accuracy'].min():.2f}%
└─ Improvement Range: {results_df['final_accuracy'].max() - results_df['final_accuracy'].min():.2f}%

🔗 VIEW RESULTS:
└─ https://wandb.ai/your-username/{PROJECT_NAME}/sweeps/{sweep_id}

📁 SAVED FILES:
├─ hyperparameter_sweep_results.csv
├─ best_hyperparameters.json
├─ hyperparameter_analysis.png
├─ 3d_hyperparameter_space.html
└─ parallel_coordinates.html
"""
    
    with open(RESULTS_DIR / 'summary_report.txt', 'w') as f:
        f.write(summary_report)
    
    print(f"📄 Summary report saved to {RESULTS_DIR / 'summary_report.txt'}")
    print(summary_report)
    
    print(f"\n✅ All results saved to: {RESULTS_DIR.absolute()}")
    print(f"🔗 View detailed results at: https://wandb.ai")
else:
    print("⚠️ No results to save. Please ensure the sweep completed successfully.")

## 🚀 Production Configuration Generator

In [ ]:
# Generate production-ready configuration
if len(results_df) > 0:
    print("🚀 Generating production configuration...")
    
    best_run = results_df.loc[results_df['final_accuracy'].idxmax()]
    
    # Create production training command
    production_cmd = f"""#!/bin/bash
# Production training command with optimized hyperparameters
# Generated from hyperparameter sweep: {sweep_id}
# Best accuracy achieved: {best_run['final_accuracy']:.2f}%

python train.py \\
    --dataset {BASE_CONFIG['dataset']} \\
    --model {BASE_CONFIG['model']} \\
    --batch-size {BASE_CONFIG['batch_size']} \\
    --diffusion-steps {BASE_CONFIG['diffusion_steps']} \\
    --supervise-energy-landscape {BASE_CONFIG['supervise_energy_landscape']} \\
    --train-num-steps {BASE_CONFIG['train_num_steps']} \\
    --save-csv-logs \\
    --csv-log-interval {BASE_CONFIG['csv_log_interval']} \\
    --use-adversarial-corruption True \\
    --anm-warmup-steps {int(best_run['warmup_steps'])} \\
    --anm-adversarial-steps {int(best_run['adversarial_steps'])} \\
    --anm-distance-penalty {float(best_run['distance_penalty']):.6f}
"""
    
    with open(RESULTS_DIR / 'production_training_command.sh', 'w') as f:
        f.write(production_cmd)
    
    # Make the script executable
    os.chmod(RESULTS_DIR / 'production_training_command.sh', 0o755)
    
    print(f"📝 Production command saved to {RESULTS_DIR / 'production_training_command.sh'}")
    
    # Create Python configuration
    python_config = {
        'adversarial_corruption': {
            'enabled': True,
            'warmup_steps': int(best_run['warmup_steps']),
            'adversarial_steps': int(best_run['adversarial_steps']),
            'distance_penalty': float(best_run['distance_penalty'])
        },
        'training': {
            'dataset': BASE_CONFIG['dataset'],
            'model': BASE_CONFIG['model'],
            'batch_size': BASE_CONFIG['batch_size'],
            'diffusion_steps': BASE_CONFIG['diffusion_steps'],
            'train_num_steps': BASE_CONFIG['train_num_steps'],
            'supervise_energy_landscape': BASE_CONFIG['supervise_energy_landscape']
        },
        'performance': {
            'expected_accuracy': float(best_run['final_accuracy']),
            'expected_loss': float(best_run['final_total_loss']),
            'sweep_id': sweep_id,
            'optimization_date': datetime.now().isoformat()
        }
    }
    
    with open(RESULTS_DIR / 'production_config.yaml', 'w') as f:
        yaml.dump(python_config, f, default_flow_style=False)
    
    print(f"⚙️ Production config saved to {RESULTS_DIR / 'production_config.yaml'}")
    
    print("\n🎯 PRODUCTION RECOMMENDATIONS:")
    print("="*50)
    print(f"✅ Use the optimized hyperparameters for production training")
    print(f"✅ Expected accuracy: {best_run['final_accuracy']:.2f}%")
    print(f"✅ Expected loss: {best_run['final_total_loss']:.4f}")
    print(f"✅ Run: bash {RESULTS_DIR / 'production_training_command.sh'}")
    print(f"✅ Or use the YAML config in your training pipeline")
else:
    print("⚠️ Cannot generate production config without results.")

## 🎉 Conclusion & Next Steps

In [ ]:
# Final summary and next steps
print("🎉 HYPERPARAMETER TUNING COMPLETED!")
print("═"*60)

if len(results_df) > 0:
    best_accuracy = results_df['final_accuracy'].max()
    worst_accuracy = results_df['final_accuracy'].min()
    improvement = best_accuracy - worst_accuracy
    
    print(f"🏆 ACHIEVED RESULTS:")
    print(f"   ├─ Best Accuracy: {best_accuracy:.2f}%")
    print(f"   ├─ Accuracy Range: {improvement:.2f}% improvement")
    print(f"   ├─ Total Runs: {len(results_df)}")
    print(f"   └─ Success Rate: {(len(results_df) / NUM_SWEEP_RUNS * 100):.1f}%")
    
    print(f"\n📁 DELIVERABLES:")
    print(f"   ├─ Optimized hyperparameters")
    print(f"   ├─ Production-ready training script")
    print(f"   ├─ Comprehensive analysis visualizations")
    print(f"   ├─ Interactive 3D parameter space")
    print(f"   └─ Statistical analysis report")
    
    print(f"\n🚀 NEXT STEPS:")
    print(f"   1. Use the optimized hyperparameters in production")
    print(f"   2. Monitor performance on your specific dataset")
    print(f"   3. Consider fine-tuning around the best parameters")
    print(f"   4. Experiment with longer training runs")
    print(f"   5. Validate results on held-out test data")
    
    print(f"\n💡 INSIGHTS:")
    best_run = results_df.loc[results_df['final_accuracy'].idxmax()]
    print(f"   ├─ Optimal warmup: {int(best_run['warmup_steps'])} steps")
    print(f"   ├─ Optimal adversarial steps: {int(best_run['adversarial_steps'])}")
    print(f"   └─ Optimal distance penalty: {float(best_run['distance_penalty']):.3f}")
    
else:
    print("⚠️ No completed runs found.")
    print("💡 Possible issues:")
    print("   ├─ Training scripts may have errors")
    print(f"   ├─ Insufficient compute time/resources")
    print(f"   ├─ Environment setup issues")
    print(f"   └─ Check wandb dashboard for detailed logs")

print(f"\n🔗 RESOURCES:")
print(f"   ├─ Weights & Biases Dashboard: https://wandb.ai")
print(f"   ├─ Results Directory: {RESULTS_DIR.absolute()}")
print(f"   └─ Project: {PROJECT_NAME}")

print(f"\n✨ Thank you for using the Adversarial Corruption Hyperparameter Tuner!")
print(f"   Happy training! 🚀")